# 04 — Model Evaluation

Comprehensive evaluation: confusion matrices, calibration, feature importances.

---

## Setup

In [ ]:
import sys, warnings
sys.path.insert(0, '..')
warnings.filterwarnings('ignore')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.calibration import calibration_curve
from sklearn.metrics import (confusion_matrix, classification_report,
                              roc_curve, auc)
from pathlib import Path

plt.rcParams.update({'figure.facecolor':'#0E1117','axes.facecolor':'#1C2130',
    'text.color':'#FAFAFA','axes.labelcolor':'#FAFAFA',
    'xtick.color':'#FAFAFA','ytick.color':'#FAFAFA','grid.color':'#2D3348'})
ACCENT = '#00FF87'
PALETTE = ['#00FF87', '#FFD700', '#FF6B6B']

from src.data_pipeline import DataPipeline
from src.feature_engineering import FeatureEngineer
from src.model import ModelTrainer, encode_labels, decode_labels, TARGET_CLASSES
from src.utils import format_probability_output
import pandas as _pd
_pd.DataFrame.to_parquet = lambda self, p, **kw: self.to_csv(str(p).replace('.parquet','.csv'), index=False)


## Load Data, Train-Test Split

In [ ]:
pipeline = DataPipeline('../data')
master = pipeline.build_master(save=False)
pipeline.load_raw_data()

complete = master[~master['is_incomplete'] & master['result'].notna()].reset_index(drop=True)

# Temporal split: last 10% as test (no leakage)
split_idx = int(len(complete) * 0.90)
train_df = complete.iloc[:split_idx].reset_index(drop=True)
test_df  = complete.iloc[split_idx:].reset_index(drop=True)
print(f"Train: {len(train_df):,} matches ({train_df['date'].min().year}–{train_df['date'].max().year})")
print(f"Test : {len(test_df):,} matches  ({test_df['date'].min().year}–{test_df['date'].max().year})")

fe = FeatureEngineer(form_windows=[5,10], master_df=master, shootouts_df=pipeline.shootouts)
fe.fit(train_df)
X_train = fe.transform(train_df)
X_test  = fe.transform(test_df)
y_train = train_df['result']
y_test  = test_df['result']
w_train = train_df['match_weight']

print(f"\nFeature matrix — Train: {X_train.shape}, Test: {X_test.shape}")


## Train & Evaluate All Models

In [ ]:
trainer = ModelTrainer(n_cv_splits=5, random_state=42)
cv_results = trainer.train_all(X_train, y_train, sample_weight=w_train, tune_hyperparams=False)

test_results = trainer.evaluate_on_test(X_test, y_test)
print(f"\nTest set — Best model ({trainer.best_model_name}):")
print(f"  Accuracy    : {test_results['accuracy']:.4f}")
print(f"  F1-weighted : {test_results['f1_weighted']:.4f}")
print(f"  Log-loss    : {test_results['log_loss']:.4f}")
print()
print("Classification report:")
for cls, metrics in test_results['classification_report'].items():
    if isinstance(metrics, dict):
        print(f"  {cls:<12} precision={metrics['precision']:.3f}  recall={metrics['recall']:.3f}  f1={metrics['f1-score']:.3f}")


## Confusion Matrix

In [ ]:
cm = np.array(test_results['confusion_matrix'])
fig, ax = plt.subplots(figsize=(7, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=TARGET_CLASSES, yticklabels=TARGET_CLASSES, ax=ax)
ax.set_title(f'Confusion Matrix — {trainer.best_model_name}', fontweight='bold', fontsize=13)
ax.set_xlabel('Predicted')
ax.set_ylabel('Actual')
plt.tight_layout()
plt.savefig('../data/processed/confusion_matrix.png', dpi=120, bbox_inches='tight')
plt.show()


## Calibration Curves

In [ ]:
y_enc = encode_labels(y_test)
y_proba = np.array(test_results['y_proba'])

fig, axes = plt.subplots(1, 3, figsize=(15, 5))
for i, (cls_name, color) in enumerate(zip(TARGET_CLASSES, PALETTE)):
    y_bin = (y_enc == i).astype(int)
    prob_cls = y_proba[:, i]
    frac_pos, mean_pred = calibration_curve(y_bin, prob_cls, n_bins=10)
    ax = axes[i]
    ax.plot([0,1],[0,1], 'k--', label='Perfect calibration')
    ax.plot(mean_pred, frac_pos, 'o-', color=color, label=cls_name)
    ax.set_title(f'Calibration — {cls_name}', fontweight='bold')
    ax.set_xlabel('Mean Predicted Probability')
    ax.set_ylabel('Fraction of Positives')
    ax.legend(fontsize=8)
    ax.grid(True, alpha=0.3)

plt.suptitle(f'Calibration Curves — {trainer.best_model_name}', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('../data/processed/calibration_curves.png', dpi=120, bbox_inches='tight')
plt.show()


## ROC Curves (One-vs-Rest)

In [ ]:
fig, ax = plt.subplots(figsize=(8, 6))
for i, (cls_name, color) in enumerate(zip(TARGET_CLASSES, PALETTE)):
    y_bin = (y_enc == i).astype(int)
    fpr, tpr, _ = roc_curve(y_bin, y_proba[:, i])
    roc_auc = auc(fpr, tpr)
    ax.plot(fpr, tpr, color=color, linewidth=2, label=f'{cls_name} (AUC={roc_auc:.3f})')

ax.plot([0,1],[0,1], 'k--', alpha=0.5)
ax.set_title(f'ROC Curves (One-vs-Rest) — {trainer.best_model_name}', fontweight='bold')
ax.set_xlabel('False Positive Rate')
ax.set_ylabel('True Positive Rate')
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('../data/processed/roc_curves.png', dpi=120, bbox_inches='tight')
plt.show()


## Feature Importances (Top 20)

In [ ]:
feat_imp = trainer.get_feature_importances(fe.get_feature_names_out())

if len(feat_imp) > 0:
    top20 = feat_imp.head(20)
    fig, ax = plt.subplots(figsize=(10, 7))
    ax.barh(top20['feature'], top20['importance'], color=ACCENT)
    ax.set_title(f'Top 20 Feature Importances — {trainer.best_model_name}', fontweight='bold')
    ax.set_xlabel('Importance')
    ax.invert_yaxis()
    plt.tight_layout()
    plt.savefig('../data/processed/feature_importances.png', dpi=120, bbox_inches='tight')
    plt.show()
    print(f"\n🔑 Most predictive feature: {top20.iloc[0]['feature']} ({top20.iloc[0]['importance']:.4f})")
else:
    print("Feature importances not available for this model type.")


## Accuracy by Tournament Tier

In [ ]:
y_pred_list = test_results['y_pred']
test_df_eval = test_df.copy().reset_index(drop=True)
test_df_eval['y_pred'] = y_pred_list
test_df_eval['correct'] = test_df_eval['result'] == test_df_eval['y_pred']

tier_acc = test_df_eval.groupby('tournament_tier')['correct'].agg(['mean','count'])
tier_acc.columns = ['accuracy','n_matches']
tier_acc['tier_name'] = tier_acc.index.map({1:'Tier 1 (Elite)',2:'Tier 2 (Qualifying)',3:'Tier 3 (Friendly)'})

print("Accuracy by Tournament Tier:")
print(tier_acc.to_string())

fig, ax = plt.subplots(figsize=(8, 5))
bars = ax.bar(tier_acc['tier_name'], tier_acc['accuracy']*100, color=PALETTE)
ax.set_title('Model Accuracy by Tournament Tier', fontweight='bold')
ax.set_ylabel('Accuracy (%)')
ax.set_ylim(0, 80)
for bar, val in zip(bars, tier_acc['accuracy']*100):
    ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.5,
            f'{val:.1f}%', ha='center', fontweight='bold', color='#FAFAFA')
plt.tight_layout()
plt.savefig('../data/processed/accuracy_by_tier.png', dpi=120, bbox_inches='tight')
plt.show()


## Brier Scores

In [ ]:
brier_scores = {}
for i, cls_name in enumerate(TARGET_CLASSES):
    y_bin = (y_enc == i).astype(int)
    prob_cls = y_proba[:, i]
    brier = np.mean((prob_cls - y_bin) ** 2)
    brier_scores[cls_name] = brier

print("Brier Scores (lower = better calibration):")
for cls, score in brier_scores.items():
    print(f"  {cls:<12}: {score:.4f}")
print(f"  Mean Brier : {np.mean(list(brier_scores.values())):.4f}")


## All Evaluation Charts Saved ✓

In [ ]:
from pathlib import Path
charts = list(Path('../data/processed').glob('*.png'))
print(f"\n{len(charts)} evaluation charts saved to data/processed/:")
for c in sorted(charts):
    print(f"  {c.name}")
